In [1]:
import pandas as pd
from konlpy.tag import Mecab
from gensim.models.doc2vec import TaggedDocument
from tqdm import tqdm
#텍스트 전처리
import re
import nltk
from nltk.corpus import stopwords

## 연극 

In [26]:
play_df=pd.read_excel("D:/연극_TOP70.xlsx")
play_df=play_df.drop(['Unnamed: 0'],axis=1)
play_df=play_df[['Title', '공연소개', '줄거리', '장르']]
play_df

,Title,공연소개,줄거리,장르
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,NaN,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,"로맨스, 코믹, 공감/힐링"
1,(리얼타임 코믹연극) 택시안에서 - 부산,NaN,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
2,(리얼타임 코믹연극) 택시안에서 - 서울,NaN,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
3,(코믹연극) 달동네-부산,NaN,NaN,"코믹, 감동, 드라마"
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"서울 종로구 창신동, 보증금 500-월세 30!\n서울 시내가 다 보이는 옥탑방!\...","이중계약으로 엮인 사랑스러운 두 청춘의, 동화 같은 사랑이 시작된다! 상큼발랄 좌충...","로맨스, 공감/힐링, 판타지"
...,...,...,...,...
65,［서울 대학로］연극 라면,NaN,수학공식처럼 복잡한 사랑에 대해 유쾌하고 코믹하게 풀어보는 90분의 시간\n때는 9...,"로맨스, 코미디, 감동"
66,［연극 on stage］ 99% - 화성,NaN,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"스릴러, 드라마, 미스터리"
67,［연극 on stage］ 띨뿌리 - 화성,NaN,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"역사극, 미스터리, 사회극"
68,［연애하기 좋은 날 : 당근거래］ - 전주,NaN,"사귄지 1년 된 지후와 시연은 ""솔로지옥"" 에서 즐겁게 데이트하다 서로에게 상처가 ...","로맨스, 코미디"


In [27]:
play_df.isna().sum()

Title     0
공연소개     47
줄거리       6
장르        0
dtype: int64

In [28]:
play_df.fillna("", inplace=True)

In [29]:
play_df['합침'] = (play_df['줄거리']) + (play_df['장르'])

In [30]:
play_df['합침'][0]

"점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 중요한 미션 과연 보늬의 운명은..? '일주일 안에 호랑이며 남자와 하룻밤을 보내지 않으면.... 세상의 모든 불운은 본인으로부터 시작된다고 믿는 운명론자 점보늬 점보늬가 사는 건물의 새건물주이자 86년생 숫총각 호랑이띠 '운명은 스스로 개척하는 것!\n제발로 걸어들어온 완벽한 부적 호랑이띠 남자\n제대리님. 첫 눈에 반한다는 말.전 믿습니다로맨스, 코믹, 공감/힐링"

In [31]:
def preprocess_text(text):
    
    # 구두점 및 특수 문자 제거
    text = re.sub(r'[^\w\s]', '', text)
    
    cleaned_text = re.sub(r'[^가-힣0-9]', ' ', text)
    
    # 토큰화 (단어 단위로 분리)
    tokens = nltk.word_tokenize(cleaned_text)
        
    # 불용어 제거    
    stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']
    filtered_tokens = [word for word in tokens if word not in stopwords]
    
    return ' '.join(filtered_tokens)


play_df['합침_cleaned'] = play_df['합침'].astype(str).apply(preprocess_text)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
play_df['합침_cleaned'] = play_df['합침_cleaned'].fillna('')

In [33]:
play_df['합침_cleaned'][0]

'점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 중요한 미션 과연 보늬의 운명은 일주일 안에 호랑이며 남자와 하룻밤을 보내지 않으면 세상의 모든 불운은 본인으로부터 시작된다고 믿는 운명론자 점보늬 점보늬가 사는 건물의 새건물주이자 86년생 숫총각 호랑이띠 운명은 스스로 개척하는 것 제발로 걸어들어온 완벽한 부적 호랑이띠 남자 제대리님 첫 눈에 반한다는 말전 믿습니다로맨스 코믹 공감힐링'

In [34]:
play_df

,Title,공연소개,줄거리,장르,합침,합침_cleaned
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,"로맨스, 코믹, 공감/힐링",점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...
1,(리얼타임 코믹연극) 택시안에서 - 부산,,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링",쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남 남들...
2,(리얼타임 코믹연극) 택시안에서 - 서울,,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링",쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남 남들...
3,(코믹연극) 달동네-부산,,,"코믹, 감동, 드라마","코믹, 감동, 드라마",코믹 감동 드라마
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"서울 종로구 창신동, 보증금 500-월세 30!\n서울 시내가 다 보이는 옥탑방!\...","이중계약으로 엮인 사랑스러운 두 청춘의, 동화 같은 사랑이 시작된다! 상큼발랄 좌충...","로맨스, 공감/힐링, 판타지","이중계약으로 엮인 사랑스러운 두 청춘의, 동화 같은 사랑이 시작된다! 상큼발랄 좌충...",이중계약으로 엮인 사랑스러운 두 청춘의 동화 같은 사랑이 시작된다 상큼발랄 좌충우돌...
...,...,...,...,...,...,...
65,［서울 대학로］연극 라면,,수학공식처럼 복잡한 사랑에 대해 유쾌하고 코믹하게 풀어보는 90분의 시간\n때는 9...,"로맨스, 코미디, 감동",수학공식처럼 복잡한 사랑에 대해 유쾌하고 코믹하게 풀어보는 90분의 시간\n때는 9...,수학공식처럼 복잡한 사랑에 대해 유쾌하고 코믹하게 풀어보는 90분의 시간 때는 90...
66,［연극 on stage］ 99% - 화성,,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"스릴러, 드라마, 미스터리",2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,2008년 세계적인 경제공황의 여파로 위기에 봉착한 중견 광고회사는 조직 리모델링을...
67,［연극 on stage］ 띨뿌리 - 화성,,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"역사극, 미스터리, 사회극",2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,2008년 세계적인 경제공황의 여파로 위기에 봉착한 중견 광고회사는 조직 리모델링을...
68,［연애하기 좋은 날 : 당근거래］ - 전주,,"사귄지 1년 된 지후와 시연은 ""솔로지옥"" 에서 즐겁게 데이트하다 서로에게 상처가 ...","로맨스, 코미디","사귄지 1년 된 지후와 시연은 ""솔로지옥"" 에서 즐겁게 데이트하다 서로에게 상처가 ...",사귄지 1년 된 지후와 시연은 솔로지옥 에서 즐겁게 데이트하다 서로에게 상처가 되는...


In [35]:
tfidf_matrix = tfidf.fit_transform(play_df['합침_cleaned'])
# overview에 대해서 tf-idf 수행
print(tfidf_matrix.shape)

(70, 2692)


In [36]:
#코사인유사도
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [37]:
indices = pd.Series(play_df.index, index=play_df['Title']).drop_duplicates()
print(indices.head())

Title
!로맨틱코미디 연극〈운빨로맨스〉! - 대구      0
(리얼타임 코믹연극) 택시안에서 - 부산       1
(리얼타임 코믹연극) 택시안에서 - 서울       2
(코믹연극) 달동네-부산                3
10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀    4
dtype: int64


In [38]:
def get_recomm(title, cosine_sim=cosine_sim):
    choice = []
    # 선택한 영화의 타이틀로부터 해당되는 인덱스를 받아옵니다. 이제 선택한 영화를 가지고 연산할 수 있습니다.
    idx = indices[title]

    # 모든 영화에 대해서 해당 영화와의 유사도를 구합니다.
    sim_scores = list(enumerate(cosine_sim[idx]))

    # 유사도에 따라 영화들을 정렬합니다.
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 가장 유사한 10개의 영화를 받아옵니다.
    sim_scores = sim_scores[1:11]

    # 가장 유사한 10개의 영화의 인덱스를 받아옵니다.
    play_indices = [i[0] for i in sim_scores]

    print('***연극 추천 순위***')
    for i in range(10):
        similar_movie_title = play_df['Title'].iloc[play_indices[i]]
        similarity_score = round(sim_scores[i][1], 2)  # 소수점 둘째 자리까지 반올림
        # 영화 제목과 해당 영화와의 코사인 유사도 출력
        print(f"{i+1}순위 : {similar_movie_title} (Cosine Similarity: {similarity_score})")


In [39]:
get_recomm('연극 〈슈만〉')

***연극 추천 순위***
1순위 : (코믹연극) 달동네-부산 (Cosine Similarity: 0.04)
2순위 : 연극 〈뷰티풀라이프〉 (Cosine Similarity: 0.04)
3순위 : !로맨틱코미디 연극〈운빨로맨스〉! - 대구 (Cosine Similarity: 0.03)
4순위 : 연극 〈운빨로맨스〉- 대학로 (Cosine Similarity: 0.03)
5순위 : 연극 〈바닷마을 다이어리〉 (Cosine Similarity: 0.03)
6순위 : ［2023 SPAF］ 연극연습 프로젝트 〈연극연습3. 극작 연습-물고기로 죽기〉 (Cosine Similarity: 0.02)
7순위 : 연극 〈나의PS파트너〉 (Cosine Similarity: 0.02)
8순위 : 괴물B (Cosine Similarity: 0.02)
9순위 : ［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 - 울산 (Cosine Similarity: 0.02)
10순위 : ［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 _ 전주 (Cosine Similarity: 0.02)


In [40]:
get_recomm('!로맨틱코미디 연극〈운빨로맨스〉! - 대구')

***연극 추천 순위***
1순위 : 연극 〈운빨로맨스〉- 대학로 (Cosine Similarity: 1.0)
2순위 : 연극 〈수상한흥신소〉 (Cosine Similarity: 0.05)
3순위 : ★평점9.5★ 코미디의맛/쇼미더퍼니 (Cosine Similarity: 0.05)
4순위 : (코믹연극) 달동네-부산 (Cosine Similarity: 0.04)
5순위 : 연극 〈쉬어 매드니스(Shear Madness)〉 - 군포 (Cosine Similarity: 0.04)
6순위 : 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀 (Cosine Similarity: 0.04)
7순위 : 연극 〈나의PS파트너〉 (Cosine Similarity: 0.04)
8순위 : 연극 〈슈만〉 (Cosine Similarity: 0.03)
9순위 : ［서울 대학로］연극 라면 (Cosine Similarity: 0.02)
10순위 : 연극 〈바닷마을 다이어리〉 (Cosine Similarity: 0.02)


In [41]:
get_recomm('(리얼타임 코믹연극) 택시안에서 - 서울')

***연극 추천 순위***
1순위 : (리얼타임 코믹연극) 택시안에서 - 서울 (Cosine Similarity: 1.0)
2순위 : 연극 한뼘사이 (Cosine Similarity: 0.06)
3순위 : 연극 〈뷰티풀라이프〉 (Cosine Similarity: 0.05)
4순위 : 연극〈늘근도둑이야기〉 (Cosine Similarity: 0.04)
5순위 : 연극 〈극적인 하룻밤〉 (Cosine Similarity: 0.04)
6순위 : 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀 (Cosine Similarity: 0.04)
7순위 : ［2023 SPAF］ 지연X전환 〈에너지_보이지 않는 언어〉 (Cosine Similarity: 0.03)
8순위 : ［연극 on stage］ 99% - 화성 (Cosine Similarity: 0.03)
9순위 : ［연극 on stage］ 띨뿌리 - 화성 (Cosine Similarity: 0.03)
10순위 : 연극 죽여주는 이야기 (Cosine Similarity: 0.03)


In [42]:
get_recomm('［서울 대학로］연극 라면')

***연극 추천 순위***
1순위 : 연극 〈수상한흥신소〉 (Cosine Similarity: 0.07)
2순위 : (코믹연극) 달동네-부산 (Cosine Similarity: 0.06)
3순위 : 연극 핫식스 (Cosine Similarity: 0.04)
4순위 : ［2023 SPAF］ 구자하/캄포 〈하마티아 3부작: 한국 연극의 역사〉 (Cosine Similarity: 0.04)
5순위 : 연극 〈뷰티풀라이프〉 (Cosine Similarity: 0.03)
6순위 : 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀 (Cosine Similarity: 0.03)
7순위 : 연극 죽여주는 이야기 (Cosine Similarity: 0.02)
8순위 : 연극 〈극적인 하룻밤〉 (Cosine Similarity: 0.02)
9순위 : !로맨틱코미디 연극〈운빨로맨스〉! - 대구 (Cosine Similarity: 0.02)
10순위 : 연극 〈운빨로맨스〉- 대학로 (Cosine Similarity: 0.02)


## 뮤지컬

In [43]:
musical_df=pd.read_excel("D:/뮤지컬_TOP70.xlsx")
musical_df=musical_df.drop(['Unnamed: 0'],axis=1)
musical_df=musical_df[['Title', '공연소개', '줄거리', '장르']]
musical_df

,Title,공연소개,줄거리,장르
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",NaN,시조'가 국가이념인 상상 속의 '조선'\n삶의 고단함과 역경을 시조 속에 담아 훌훌...,"역사극, 판타지"
1,2023 이은결 더 일루션 마스터피스 - 이천,NaN,NaN,"마술, 판타지"
2,2023 창작뮤지컬어워드 NEXT,None,NaN,드라마
3,2023 최현우 Answer - 대구,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
4,2023 최현우 Answer - 성남,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
...,...,...,...,...
64,창작가무극 〈순신〉,최고의 창작진이 선보이는 서울예술단 창작 신작!\n국공립단체로서 한국적 콘텐츠의 가...,"1592년 임진년 4월 어느 날, 순신은 불길한 꿈을 꾼다.\n그의 꿈처럼 임진왜란...","역사극, 드라마"
65,태양의서커스 〈루치아〉,NaN,노란마리골드 꽃밭에 추락한 낙하산\n호기심에 거대한 금속 열쇠를 돌리자 펼쳐지는 꿈...,"판타지, 액션"
66,［뮤지컬 on stage］ 뮤지컬 〈쇼맨 : 어느 독재자의 네 번째 대역배우〉 - 화성,NaN,NaN,"드라마, 공감/힐링"
67,［제주］뉴미디어뮤지컬(노형수퍼마켙),NaN,"시간의 여신, 오늘\n노형수퍼마켙만의 새로운 상설 공연으로 지루할 틈 없이 다양한 ...","판타지, 액션"


In [44]:
musical_df.fillna("", inplace=True)

In [46]:
musical_df['합침'] = (musical_df['줄거리']) + (musical_df['장르'])

In [47]:
def preprocess_text(text):
    
    # 구두점 및 특수 문자 제거
    text = re.sub(r'[^\w\s]', '', text)
    
    cleaned_text = re.sub(r'[^가-힣0-9]', ' ', text)
    
    # 토큰화 (단어 단위로 분리)
    tokens = nltk.word_tokenize(cleaned_text)
    
    
    # 불용어 제거    
    stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']
    filtered_tokens = [word for word in tokens if word not in stopwords]
    
    return ' '.join(filtered_tokens)


musical_df['합침_cleaned'] = musical_df['합침'].astype(str).apply(preprocess_text)

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
musical_df['합침_cleaned'] = musical_df['합침_cleaned'].fillna('')

In [49]:
#코사인유사도
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [50]:
indices = pd.Series(musical_df.index, index=musical_df['Title']).drop_duplicates()
print(indices.head())

Title
2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천    0
2023 이은결 더 일루션 마스터피스 - 이천      1
2023 창작뮤지컬어워드 NEXT             2
2023 최현우 Answer - 대구           3
2023 최현우 Answer - 성남           4
dtype: int64


In [51]:
def get_recomm(title, cosine_sim=cosine_sim):
    choice = []
    # 선택한 영화의 타이틀로부터 해당되는 인덱스를 받아옵니다. 이제 선택한 영화를 가지고 연산할 수 있습니다.
    idx = indices[title]

    # 모든 영화에 대해서 해당 영화와의 유사도를 구합니다.
    sim_scores = list(enumerate(cosine_sim[idx]))

    # 유사도에 따라 영화들을 정렬합니다.
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 가장 유사한 10개의 영화를 받아옵니다.
    sim_scores = sim_scores[1:11]

    # 가장 유사한 10개의 영화의 인덱스를 받아옵니다.
    musical_indices = [i[0] for i in sim_scores]

    print('***연극 추천 순위***')
    for i in range(10):
        similar_musical_title = musical_df['Title'].iloc[musical_indices[i]]
        similarity_score = round(sim_scores[i][1], 2)  # 소수점 둘째 자리까지 반올림
        # 영화 제목과 해당 영화와의 코사인 유사도 출력
        print(f"{i+1}순위 : {similar_musical_title} (Cosine Similarity: {similarity_score})")


In [79]:
get_recomm('태양의서커스 〈루치아〉')

***연극 추천 순위***
1순위 : 태양의서커스 〈루치아〉 (Cosine Similarity: 1.0)
2순위 : 뮤지컬 〈김종욱 찾기〉 (Cosine Similarity: 0.07)
3순위 : 2023 최현우 Answer - 대구 (Cosine Similarity: 0.06)
4순위 : 뮤지컬 〈맘마미아〉 - 여수 (Cosine Similarity: 0.04)
5순위 : 뮤지컬 〈오페라의 유령〉 - 서울 (Cosine Similarity: 0.04)
6순위 : 뮤지컬 〈그날들〉 10주년 기념 공연 - 대구 (Cosine Similarity: 0.03)
7순위 : 2023 최현우 Answer - 성남 (Cosine Similarity: 0.03)
8순위 : 뮤지컬 〈맘마미아!〉 - 김해 (Cosine Similarity: 0.02)
9순위 : 뮤지컬 〈곤 투모로우〉 (Cosine Similarity: 0.02)
10순위 : 2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천 (Cosine Similarity: 0.02)


In [80]:
get_recomm('［청주,세종］뮤지컬 빌의구둣방')

***연극 추천 순위***
1순위 : ［청주,세종］뮤지컬 빌의구둣방 (Cosine Similarity: 1.0)
2순위 : 뮤지컬 〈렌트〉 (Cosine Similarity: 0.04)
3순위 : 2023 최현우 Answer - 안산 (Cosine Similarity: 0.04)
4순위 : 2023 최현우 Answer - 울산 (Cosine Similarity: 0.04)
5순위 : 2023~24 최현우 Answer (Cosine Similarity: 0.04)
6순위 : 2023송승환의 오리지널 난타(대전) (Cosine Similarity: 0.04)
7순위 : 뮤지컬 〈칠칠〉 (Cosine Similarity: 0.04)
8순위 : 뮤지컬 〈그날들〉 10주년 기념 공연 - 대구 (Cosine Similarity: 0.03)
9순위 : 뮤지컬 〈빨래〉 (Cosine Similarity: 0.03)
10순위 : 2023 이은결 더 일루션 마스터피스 - 이천 (Cosine Similarity: 0.02)
